#### Cluster profiling - using assigned cluster values to create new variables - version 2, support variables
EDIT: using cluster profiling for the target variable can be considered a soft target leakage in certain circumstances.

This means, if future data that will be used with the model follows unseen trends => the original target -based cluster profiling no longer works in reality, and has to be re-calculated.

Therefore it might be better to use cluster profiling for support variables only (independent variables).

Using cluster profiling for target variable is very similar to target encoding => adding a statistically descriptive new variable for the expected target.

In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# load data
df = pd.read_csv("HeartDisease_cleaned.csv")
df

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST
0,40,1,140,289,0,172,0,0.0,3,0,0,1,0,0,0,1,0
1,49,0,160,180,0,156,0,1.0,2,1,0,0,1,0,0,1,0
2,37,1,130,283,0,98,0,0.0,3,0,0,1,0,0,0,0,1
3,48,0,138,214,0,108,1,1.5,2,1,1,0,0,0,0,1,0
4,54,1,150,195,0,122,0,0.0,3,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,1,110,264,0,132,0,1.2,2,1,0,0,0,1,0,1,0
914,68,1,144,193,1,141,0,3.4,2,1,1,0,0,0,0,1,0
915,57,1,130,131,0,115,1,1.2,2,1,1,0,0,0,0,1,0
916,57,0,130,236,0,174,0,0.0,2,1,0,1,0,0,1,0,0


In [2]:
# X/y -split
X = df.drop("HeartDisease", axis=1)

# most clustering algorithms require scaled values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# create clustering model and train it with our data
kmeans = KMeans(n_clusters=6, random_state=42)

# place the cluster value back to DataFrame
df['cluster'] = kmeans.fit_predict(X_scaled)

df.head(5)

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,cluster
0,40,1,140,289,0,172,0,0.0,3,0,0,1,0,0,0,1,0,2
1,49,0,160,180,0,156,0,1.0,2,1,0,0,1,0,0,1,0,4
2,37,1,130,283,0,98,0,0.0,3,0,0,1,0,0,0,0,1,2
3,48,0,138,214,0,108,1,1.5,2,1,1,0,0,0,0,1,0,5
4,54,1,150,195,0,122,0,0.0,3,0,0,0,1,0,0,1,0,4


In [3]:
# let's how the cluster values are spread out
df['cluster'].value_counts()

cluster
5    221
4    173
2    164
0    163
1    151
3     46
Name: count, dtype: int64

In [5]:
# let's see how the clusters are spread around the target variable (0 = no heart disease, 1 = heart disease)
# this is only for studying the clusters, it does not create any new variable
cluster_counts = df.groupby(['HeartDisease', 'cluster']).size().unstack(fill_value=0)
print(cluster_counts)


# remember to lock down the random seed when you find a good clustering result

# ideally the clustering should clearly separate different qualities in the target
# currently, it's a bit all over the place

# also remember the original data is mostly concentrated on qualities 5 and 6
# => this is why most values are in qualities 5 and 6

cluster         0   1    2   3    4    5
HeartDisease                            
0              24  54  147  26  127   32
1             139  97   17  20   46  189


#### Perform cluster profiling => calculate averages based on the created clusters

In [6]:
# calculate the average of maximum heart rate for each cluster
cluster_maxhr_average = df.groupby('cluster')['MaxHR'].mean()

# place the averages based on clusters back into DataFrame
df['cluster_average_maxhr'] = df['cluster'].map(cluster_maxhr_average)

df.head(10)

,Age,Sex,RestingBP,Cholesterol,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,cluster,cluster_average_maxhr
0,40,1,140,289,0,172,0,0.0,3,0,0,1,0,0,0,1,0,2,152.006098
1,49,0,160,180,0,156,0,1.0,2,1,0,0,1,0,0,1,0,4,150.393064
2,37,1,130,283,0,98,0,0.0,3,0,0,1,0,0,0,0,1,2,152.006098
3,48,0,138,214,0,108,1,1.5,2,1,1,0,0,0,0,1,0,5,124.076923
4,54,1,150,195,0,122,0,0.0,3,0,0,0,1,0,0,1,0,4,150.393064
5,39,1,120,339,0,170,0,0.0,3,0,0,0,1,0,0,1,0,4,150.393064
6,45,0,130,237,0,170,0,0.0,3,0,0,1,0,0,0,1,0,2,152.006098
7,54,1,110,208,0,142,0,0.0,3,0,0,1,0,0,0,1,0,2,152.006098
8,37,1,140,207,0,130,1,1.5,2,1,1,0,0,0,0,1,0,5,124.076923
9,48,0,120,284,0,120,0,0.0,3,0,0,1,0,0,0,1,0,2,152.006098


In [7]:
# study the average maximum heart rate for each cluster
# check how well the clusters are separated, and if some clusters are too close
df.groupby("cluster", as_index=False)["cluster_average_maxhr"].mean()

,cluster,cluster_average_maxhr
0,0,119.723926
1,1,138.443709
2,2,152.006098
3,3,147.891304
4,4,150.393064
5,5,124.076923
